# Landslide source/runout step 08: 50-year discounted avoided EAD summary

This notebook applies the same discount-factor method used in the river flooding and coastal flooding analyses to the landslide source/runout EAD outputs.

Assumptions:

- annual EADs are constant over time;
- present value is calculated over `50` years;
- discount rate is `10%`;
- the discount factor is computed as `sum(1 / (1 + r) ** year for year in range(years + 1))`.

This matches the coastal flooding long-timeframe notebook, including **year 0** in the discount-factor sum.

The notebook calculates discounted present values for:

- avoided EAD from protecting existing forests: `Deforestation EAD - Baseline EAD`;
- avoided EAD from reafforestation: `Baseline EAD - Reafforestation EAD`;
- localized increases in damages where those differences are negative at asset level.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


In [ ]:
base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
landslide_results_root = base_path / 'dphil_paper_3/results/02_damage_estimates/landslide_damages'

scenario_paths = {
    'minimum': landslide_results_root / 'results_landslide_minimum_scenario_source_and_runout_zones/damage_estimates',
    'maximum': landslide_results_root / 'results_landslide_maximum_scenario_source_and_runout_zones/damage_estimates',
}

combined_output_dir = landslide_results_root / 'discounted_50_year_source_and_runout_zones'
combined_output_dir.mkdir(parents=True, exist_ok=True)

for scenario_name, damage_estimates_dir in scenario_paths.items():
    if not damage_estimates_dir.exists():
        raise FileNotFoundError(f'Missing damage estimates directory for {scenario_name}: {damage_estimates_dir}')
    required_files = [
        damage_estimates_dir / 'landslide_ead_asset_level_usd.csv',
        damage_estimates_dir / 'landslide_ead_sector_summary_usd.csv',
        damage_estimates_dir / 'landslide_ead_subsector_summary_usd.csv',
        damage_estimates_dir / 'landslide_ead_overall_totals_usd.csv',
    ]
    for required_file in required_files:
        if not required_file.exists():
            raise FileNotFoundError(f'Missing required file: {required_file}')

print('Combined output directory:', combined_output_dir)


In [ ]:
discount_years = 50
discount_rate = 0.10
discounted_suffix = 'PV_50Y_10pct'


def compute_discount_factor(years: int, annual_discount_rate: float) -> float:
    return sum(1 / (1 + annual_discount_rate) ** year for year in range(years + 1))


def format_usd_readable(value):
    if pd.isna(value):
        return 'NA'
    value = float(value)
    abs_value = abs(value)
    sign = '-' if value < 0 else ''
    if abs_value >= 1_000_000_000:
        return f'{sign}US${abs_value / 1_000_000_000:,.2f} billion'
    if abs_value >= 1_000_000:
        return f'{sign}US${abs_value / 1_000_000:,.2f} million'
    if abs_value >= 1_000:
        return f'{sign}US${abs_value / 1_000:,.1f} thousand'
    return f'{sign}US${abs_value:,.0f}'


def format_pct(value):
    if pd.isna(value):
        return 'NA'
    return f'{float(value):,.2f}%'


discount_factor_50_years = compute_discount_factor(discount_years, discount_rate)

metadata = pd.DataFrame([
    {
        'Discount_Years': discount_years,
        'Discount_Rate': discount_rate,
        'Discount_Factor': discount_factor_50_years,
        'Method': 'sum(1 / (1 + discount_rate) ** year for year in range(years + 1))',
        'Includes_Year_0': True,
    }
])
metadata_file = combined_output_dir / 'landslide_50_year_discount_factor_10pct.csv'
metadata.to_csv(metadata_file, index=False)

print('Discount factor:', discount_factor_50_years)
print('Saved:', metadata_file)
metadata


In [ ]:
benefit_columns = [
    'Protection_Avoided_EAD_USD',
    'Reafforestation_Avoided_EAD_USD',
    'Protection_Increased_Damage_USD',
    'Reafforestation_Increased_Damage_USD',
    'Protection_Gross_Avoided_EAD_USD',
    'Reafforestation_Gross_Avoided_EAD_USD',
]


def add_discounted_columns(table: pd.DataFrame, annual_columns: list[str]) -> pd.DataFrame:
    out = table.copy()
    for col in annual_columns:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors='coerce').fillna(0.0)
            out[f'{col}_{discounted_suffix}'] = out[col] * discount_factor_50_years
            out[f'{col}_Readable'] = out[col].apply(format_usd_readable)
            out[f'{col}_{discounted_suffix}_Readable'] = out[f'{col}_{discounted_suffix}'].apply(format_usd_readable)
    return out


def add_percent_labels(table: pd.DataFrame) -> pd.DataFrame:
    out = table.copy()
    pct_cols = [col for col in out.columns if col.endswith('_Pct') or col.endswith('_Share')]
    for col in pct_cols:
        out[f'{col}_Label'] = out[col].apply(format_pct)
    return out


def build_asset_metrics(asset_ead: pd.DataFrame) -> pd.DataFrame:
    required = [
        'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID',
        'EAD_Baseline_USD', 'EAD_Deforestation_USD', 'EAD_Reafforestation_USD',
    ]
    missing = [col for col in required if col not in asset_ead.columns]
    if missing:
        raise KeyError(f'Asset EAD table missing columns: {missing}')

    out = asset_ead[required].copy()
    out['Protection_Net_Avoided_EAD_USD'] = out['EAD_Deforestation_USD'] - out['EAD_Baseline_USD']
    out['Reafforestation_Net_Avoided_EAD_USD'] = out['EAD_Baseline_USD'] - out['EAD_Reafforestation_USD']

    out['Protection_Avoided_EAD_USD'] = out['Protection_Net_Avoided_EAD_USD']
    out['Reafforestation_Avoided_EAD_USD'] = out['Reafforestation_Net_Avoided_EAD_USD']

    out['Protection_Gross_Avoided_EAD_USD'] = out['Protection_Net_Avoided_EAD_USD'].clip(lower=0.0)
    out['Reafforestation_Gross_Avoided_EAD_USD'] = out['Reafforestation_Net_Avoided_EAD_USD'].clip(lower=0.0)

    out['Protection_Increased_Damage_USD'] = (-out['Protection_Net_Avoided_EAD_USD']).clip(lower=0.0)
    out['Reafforestation_Increased_Damage_USD'] = (-out['Reafforestation_Net_Avoided_EAD_USD']).clip(lower=0.0)

    return add_discounted_columns(out, benefit_columns + ['Protection_Net_Avoided_EAD_USD', 'Reafforestation_Net_Avoided_EAD_USD'])


def aggregate_metrics(asset_metrics: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    annual_cols = benefit_columns + [
        'Protection_Net_Avoided_EAD_USD',
        'Reafforestation_Net_Avoided_EAD_USD',
        'EAD_Baseline_USD',
        'EAD_Deforestation_USD',
        'EAD_Reafforestation_USD',
    ]
    summary = (
        asset_metrics
        .groupby(group_cols, as_index=False)[annual_cols]
        .sum()
    )
    summary['Protection_Avoided_Pct_vs_Deforestation'] = np.where(
        summary['EAD_Deforestation_USD'] > 0,
        100.0 * summary['Protection_Avoided_EAD_USD'] / summary['EAD_Deforestation_USD'],
        np.nan,
    )
    summary['Protection_Avoided_Pct_vs_Baseline'] = np.where(
        summary['EAD_Baseline_USD'] > 0,
        100.0 * summary['Protection_Avoided_EAD_USD'] / summary['EAD_Baseline_USD'],
        np.nan,
    )
    summary['Reafforestation_Avoided_Pct_vs_Baseline'] = np.where(
        summary['EAD_Baseline_USD'] > 0,
        100.0 * summary['Reafforestation_Avoided_EAD_USD'] / summary['EAD_Baseline_USD'],
        np.nan,
    )

    discounted_cols = benefit_columns + [
        'Protection_Net_Avoided_EAD_USD',
        'Reafforestation_Net_Avoided_EAD_USD',
    ]
    summary = add_discounted_columns(summary, discounted_cols)
    summary = add_percent_labels(summary)
    return summary


In [ ]:
discounted_tables_by_scenario = {}

for scenario_name, damage_estimates_dir in scenario_paths.items():
    scenario_output_dir = damage_estimates_dir / 'discounted_50_year_source_and_runout_zones'
    scenario_output_dir.mkdir(parents=True, exist_ok=True)

    asset_ead = pd.read_csv(damage_estimates_dir / 'landslide_ead_asset_level_usd.csv', low_memory=False)
    asset_metrics = build_asset_metrics(asset_ead)

    total_summary = aggregate_metrics(asset_metrics.assign(Group='Total'), ['Group'])
    sector_summary = aggregate_metrics(asset_metrics, ['Sector'])
    subsector_summary = aggregate_metrics(asset_metrics, ['Sector', 'Subsector'])

    for table in [total_summary, sector_summary, subsector_summary]:
        table.insert(0, 'Damage_Case', scenario_name)
        table['Discount_Years'] = discount_years
        table['Discount_Rate'] = discount_rate
        table['Discount_Factor'] = discount_factor_50_years

    asset_metrics.insert(0, 'Damage_Case', scenario_name)
    asset_metrics['Discount_Years'] = discount_years
    asset_metrics['Discount_Rate'] = discount_rate
    asset_metrics['Discount_Factor'] = discount_factor_50_years

    total_file = scenario_output_dir / 'landslide_ead_total_summary_annual_and_50yr_discounted_10pct.csv'
    sector_file = scenario_output_dir / 'landslide_ead_sector_summary_annual_and_50yr_discounted_10pct.csv'
    subsector_file = scenario_output_dir / 'landslide_ead_subsector_summary_annual_and_50yr_discounted_10pct.csv'
    asset_file = scenario_output_dir / 'landslide_ead_asset_level_annual_and_50yr_discounted_10pct.csv'

    total_summary.to_csv(total_file, index=False)
    sector_summary.to_csv(sector_file, index=False)
    subsector_summary.to_csv(subsector_file, index=False)
    asset_metrics.to_csv(asset_file, index=False)

    discounted_tables_by_scenario[scenario_name] = {
        'total': total_summary,
        'sector': sector_summary,
        'subsector': subsector_summary,
        'asset': asset_metrics,
    }

    print(f'[{scenario_name}] Saved:', total_file)
    print(f'[{scenario_name}] Saved:', sector_file)
    print(f'[{scenario_name}] Saved:', subsector_file)
    print(f'[{scenario_name}] Saved:', asset_file)


In [ ]:
combined_total = pd.concat(
    [tables['total'] for tables in discounted_tables_by_scenario.values()],
    ignore_index=True,
)
combined_sector = pd.concat(
    [tables['sector'] for tables in discounted_tables_by_scenario.values()],
    ignore_index=True,
)
combined_subsector = pd.concat(
    [tables['subsector'] for tables in discounted_tables_by_scenario.values()],
    ignore_index=True,
)

combined_total_file = combined_output_dir / 'landslide_ead_total_summary_annual_and_50yr_discounted_10pct_min_max.csv'
combined_sector_file = combined_output_dir / 'landslide_ead_sector_summary_annual_and_50yr_discounted_10pct_min_max.csv'
combined_subsector_file = combined_output_dir / 'landslide_ead_subsector_summary_annual_and_50yr_discounted_10pct_min_max.csv'

combined_total.to_csv(combined_total_file, index=False)
combined_sector.to_csv(combined_sector_file, index=False)
combined_subsector.to_csv(combined_subsector_file, index=False)

print('Saved:', combined_total_file)
print('Saved:', combined_sector_file)
print('Saved:', combined_subsector_file)

display(combined_total)


In [ ]:
def min_max_range(table: pd.DataFrame, group_cols: list[str], value_cols: list[str]) -> pd.DataFrame:
    rows = []
    for key, group in table.groupby(group_cols, dropna=False):
        if not isinstance(key, tuple):
            key = (key,)
        row = dict(zip(group_cols, key))
        for col in value_cols:
            row[f'{col}_Min'] = group[col].min()
            row[f'{col}_Max'] = group[col].max()
            row[f'{col}_Min_Readable'] = format_usd_readable(group[col].min()) if 'USD' in col else format_pct(group[col].min())
            row[f'{col}_Max_Readable'] = format_usd_readable(group[col].max()) if 'USD' in col else format_pct(group[col].max())
        rows.append(row)
    return pd.DataFrame(rows)

range_value_cols = [
    f'Protection_Avoided_EAD_USD_{discounted_suffix}',
    f'Reafforestation_Avoided_EAD_USD_{discounted_suffix}',
    f'Protection_Increased_Damage_USD_{discounted_suffix}',
    f'Reafforestation_Increased_Damage_USD_{discounted_suffix}',
]

sector_range = min_max_range(combined_sector, ['Sector'], range_value_cols)
subsector_range = min_max_range(combined_subsector, ['Sector', 'Subsector'], range_value_cols)
total_range = min_max_range(combined_total, ['Group'], range_value_cols)

sector_range_file = combined_output_dir / 'landslide_ead_sector_50yr_discounted_10pct_ranges.csv'
subsector_range_file = combined_output_dir / 'landslide_ead_subsector_50yr_discounted_10pct_ranges.csv'
total_range_file = combined_output_dir / 'landslide_ead_total_50yr_discounted_10pct_ranges.csv'

sector_range.to_csv(sector_range_file, index=False)
subsector_range.to_csv(subsector_range_file, index=False)
total_range.to_csv(total_range_file, index=False)

print('Saved:', total_range_file)
print('Saved:', sector_range_file)
print('Saved:', subsector_range_file)

display(total_range)
display(sector_range.sort_values(f'Protection_Avoided_EAD_USD_{discounted_suffix}_Max', ascending=False))


In [ ]:
# Compact tables for manuscript drafting.
summary_cols = [
    'Damage_Case',
    'Protection_Avoided_EAD_USD',
    f'Protection_Avoided_EAD_USD_{discounted_suffix}',
    'Protection_Increased_Damage_USD',
    f'Protection_Increased_Damage_USD_{discounted_suffix}',
    'Reafforestation_Avoided_EAD_USD',
    f'Reafforestation_Avoided_EAD_USD_{discounted_suffix}',
    'Reafforestation_Increased_Damage_USD',
    f'Reafforestation_Increased_Damage_USD_{discounted_suffix}',
]

print('National totals:')
display(combined_total[summary_cols])

print('Sector totals:')
display(combined_sector[['Damage_Case', 'Sector'] + summary_cols[1:]].sort_values(['Sector', 'Damage_Case']))

print('Top subsectors by protection PV benefit:')
display(
    combined_subsector[
        ['Damage_Case', 'Sector', 'Subsector'] + summary_cols[1:]
    ]
    .sort_values(f'Protection_Avoided_EAD_USD_{discounted_suffix}', ascending=False)
    .head(20)
)

print('Top subsectors by reafforestation PV benefit:')
display(
    combined_subsector[
        ['Damage_Case', 'Sector', 'Subsector'] + summary_cols[1:]
    ]
    .sort_values(f'Reafforestation_Avoided_EAD_USD_{discounted_suffix}', ascending=False)
    .head(20)
)


In [ ]:
# Generate a short machine-readable drafting table with readable strings.
drafting_rows = []
for table_name, table, label_cols in [
    ('total', combined_total, ['Group']),
    ('sector', combined_sector, ['Sector']),
    ('subsector', combined_subsector, ['Sector', 'Subsector']),
]:
    out = table.copy()
    out['Table'] = table_name
    for col in summary_cols[1:]:
        out[f'{col}_Readable'] = out[col].apply(format_usd_readable)
    drafting_rows.append(out[['Table', 'Damage_Case'] + label_cols + [f'{col}_Readable' for col in summary_cols[1:]]])

drafting_table = pd.concat(drafting_rows, ignore_index=True, sort=False)
drafting_table_file = combined_output_dir / 'landslide_50yr_discounted_10pct_drafting_values_readable.csv'
drafting_table.to_csv(drafting_table_file, index=False)
print('Saved:', drafting_table_file)
display(drafting_table.head(30))
